# 1. Khởi tạo môi trường & Đọc dữ liệu

In [17]:
# export
import json
import random
import re
import time
from urllib.parse import quote

import pandas as pd
import requests
from pytrends.request import TrendReq

from helpers.get_configs import get_config

MAJORS_PATH = get_config('dirs.yaml', ['cleaned_data', 'majors'], as_path=True)
CUTOFF_SCORES_PATH = get_config('dirs.yaml', ['cleaned_data', 'cutoff_scores'], as_path=True)
SCORE_BLOCKS_PATH = get_config('dirs.yaml', ['cleaned_data', 'score_blocks'], as_path=True)
TOPICS_PATH = get_config('dirs.yaml', ['raw_data', 'trend_topics'], as_path=True)
SCRIPT_OUTPUT_PATH = get_config('dirs.yaml', ['unit_scripts', 'get_topics'], as_path=True)

df_majors = pd.read_csv(MAJORS_PATH, encoding='utf-8', usecols=['major_group_name', 'major_name'])
df_cutoff = pd.read_csv(CUTOFF_SCORES_PATH, encoding='utf-8', usecols=['school_name', 'score_admission_name'])
df_blocks = pd.read_csv(SCORE_BLOCKS_PATH, encoding='utf-8')

In [18]:
# Khảo sát thông tin tổng quan của dữ liệu
df_majors.info()
df_cutoff.info()
df_blocks.info()
df_blocks.head()

<class 'pandas.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   major_group_name  506 non-null    str  
 1   major_name        506 non-null    str  
dtypes: str(2)
memory usage: 38.2 KB
<class 'pandas.DataFrame'>
RangeIndex: 101466 entries, 0 to 101465
Data columns (total 2 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   school_name           101466 non-null  str  
 1   score_admission_name  101466 non-null  str  
dtypes: str(2)
memory usage: 7.3 MB
<class 'pandas.DataFrame'>
RangeIndex: 280 entries, 0 to 279
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   code      280 non-null    str  
 1   subject1  280 non-null    str  
 2   subject2  280 non-null    str  
 3   subject3  280 non-null    str  
 4   subject4  1 non-null      str  
dty

,code,subject1,subject2,subject3,subject4
0,D01,Toán,Ngữ văn,Tiếng anh,NaN
1,A00,Toán,Vật lí,Hóa học,NaN
2,A01,Toán,Vật lí,Tiếng anh,NaN
3,D07,Toán,Hóa học,Tiếng anh,NaN
4,B00,Toán,Hóa học,Sinh học,NaN


# 2. Thành lập danh sách unique

In [19]:
# export
def unique_values(series, sep=None):
    """Chuẩn hóa khoảng trắng, bỏ rỗng/NaN và loại trùng (không phân biệt hoa/thường), giữ nguyên thứ tự xuất hiện.
    Nếu truyền `sep`, mỗi ô sẽ được tách theo dấu phân cách đó trước khi lấy unique."""
    series = series.dropna().astype(str)
    if sep:
        series = series.str.split(sep).explode()
    cleaned = series.astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
    seen, result = set(), []
    for value in cleaned:
        key = value.casefold()
        if value and key not in seen:
            seen.add(key)
            result.append(value)
    return result


school_names = unique_values(df_cutoff['school_name'])
major_group_names = unique_values(df_majors['major_group_name'])
major_names = unique_values(df_majors['major_name'], sep=';')   # major_name phân cách bằng ';'
admission_method_names = unique_values(df_cutoff['score_admission_name'])

## Tổ hợp môn (subject_group) & môn học (subject)

- `subject_group_name` = `subject1 - subject2 - subject3 - subject4` (bỏ qua ô rỗng), ví dụ `Toán - Vật lí - Hóa học` hoặc `Toán - Vật lí - Hóa học - Sinh học`.
- `subject_names` = unique của toàn bộ các cột `subject1..subject4`.

In [20]:
# export
SUBJECT_COLUMNS = ['subject1', 'subject2', 'subject3', 'subject4']
SUBJECT_GROUP_COLUMNS = ['subject1', 'subject2', 'subject3', 'subject4']


def build_subject_group_name(row):
    parts = [re.sub(r'\s+', ' ', str(row[c])).strip() for c in SUBJECT_GROUP_COLUMNS if pd.notna(row[c])]
    parts = [p for p in parts if p]
    return ' - '.join(parts)


subject_group_names = unique_values(df_blocks.apply(build_subject_group_name, axis=1))
subject_names = unique_values(df_blocks[SUBJECT_COLUMNS].stack())   # stack() tự bỏ NaN

print(f"school_names:           {len(school_names)}")
print(f"major_group_names:      {len(major_group_names)}")
print(f"major_names:            {len(major_names)}")
print(f"admission_method_names: {len(admission_method_names)}")
print(f"subject_group_names:    {len(subject_group_names)}")
print(f"subject_names:          {len(subject_names)}")

school_names:           302
major_group_names:      36
major_names:            451
admission_method_names: 13
subject_group_names:    274
subject_names:          80


In [21]:
# Xem nhanh các danh sách
for name, values in [('school', school_names), ('major_group', major_group_names),
                     ('major', major_names), ('admission_method', admission_method_names),
                     ('subject_group', subject_group_names), ('subject', subject_names)]:
    print(f"--- {name} ---")
    print(values[:5])

--- school ---
['Trường Cao Đẳng Kỹ Thuật Cao Thắng', 'Trường Cao Đẳng Công Thương TPHCM', 'Đại Học Kinh Tế Quốc Dân', 'Đại Học Bách Khoa Hà Nội', 'Học Viện Tài chính']
--- major_group ---
['Kế toán - Kiểm toán', 'Tài chính - Ngân hàng - Bảo hiểm', 'Kinh tế - Quản trị kinh doanh - Thương Mại', 'Công nghệ thông tin - Tin học', 'Công nghiệp bán dẫn']
--- major ---
['Quản trị tài chính kế toán', 'Kế toán', 'Kiểm toán', 'Kinh tế và quản lý bất động sản', 'Tài chính']
--- admission_method ---
['Điểm thi THPT', 'Điểm học bạ', 'Điểm ĐGNL V-ACT', 'Điểm ĐGNL HSA', 'Điểm ĐGTD TSA']
--- subject_group ---
['Toán - Ngữ văn - Tiếng anh', 'Toán - Vật lí - Hóa học', 'Toán - Vật lí - Tiếng anh', 'Toán - Hóa học - Tiếng anh', 'Toán - Hóa học - Sinh học']
--- subject ---
['Toán', 'Ngữ văn', 'Tiếng anh', 'Vật lí', 'Hóa học']


# 3. Cấu hình & cơ chế lấy topic (có fallback)

Chuỗi fallback cho mỗi keyword:

1. `pytrends.suggestions()` — thử lại tối đa `MAX_RETRIES` lần (timeout + exponential backoff có jitter).
2. Nếu vẫn lỗi: **gửi request thủ công** bằng `requests.Session` tới đúng endpoint autocomplete của Google Trends (cùng `hl=vi-VN`, `tz=-420`), cũng thử lại tối đa `MAX_RETRIES` lần.
3. Mỗi lần chuyển bước hoặc gặp HTTP 429 sẽ **tạo lại session/cookie**.
4. Nếu liên tiếp nhiều keyword thất bại thì nghỉ dài (`COOLDOWN_SECONDS`) rồi chạy tiếp.

> Lưu ý: endpoint autocomplete của Google Trends **không có tham số `geo`**, nên "địa điểm Việt Nam" được áp dụng qua `hl='vi-VN'` và `tz=-420` (UTC+7).

In [22]:
# export
# --- Địa điểm / ngôn ngữ: Việt Nam ---
HL = 'vi-VN'
TZ = -420                      # UTC+7, tính theo phút (pytrends dùng giá trị âm cho múi giờ phía Đông)

# --- Timeout / retry ---
REQUEST_TIMEOUT = (5, 15)      # (connect, read) giây
MAX_RETRIES = 3                # số lần thử cho MỖI phương thức (pytrends, thủ công)
BACKOFF_BASE = 2               # chờ = BACKOFF_BASE ** lần_thử + jitter
RATE_LIMIT_MULTIPLIER = 2      # nhân thời gian chờ khi gặp 429
SLEEP_RANGE = (1.0, 2.5)       # nghỉ ngẫu nhiên giữa 2 keyword thành công
CONSECUTIVE_FAIL_LIMIT = 3
COOLDOWN_SECONDS = 60

SUGGESTIONS_URL = 'https://trends.google.com/trends/api/autocomplete/'
OUTPUT_COLUMNS = ['keyword', 'keyword_type', 'mid', 'title', 'type']
USER_AGENT = ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
              '(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36')

In [23]:
# export
class FetchError(Exception):
    """Mọi phương thức/lần thử đều thất bại cho một keyword."""


def _is_rate_limited(error):
    msg = str(error)
    return '429' in msg or 'Too Many Requests' in msg


class TopicFetcher:
    def __init__(self):
        self.preferred = 'pytrends'   # phương thức vừa thành công gần nhất sẽ được thử trước
        self._reset()

    # ---------- khởi tạo / làm mới session ----------
    def _reset(self):
        try:
            # Lưu ý: không truyền retries/backoff_factor vào TrendReq vì pytrends cũ
            # lỗi với urllib3>=2 (method_whitelist). Tự retry ở bên dưới.
            self.pytrends = TrendReq(hl=HL, tz=TZ, timeout=REQUEST_TIMEOUT)
        except Exception as e:
            print(f"  [warn] Không khởi tạo được TrendReq: {e}")
            self.pytrends = None

        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': USER_AGENT,
            'Accept-Language': 'vi-VN,vi;q=0.9,en;q=0.5',
            'Accept': 'application/json, text/plain, */*',
        })
        try:  # lấy cookie (NID...) giống trình duyệt
            self.session.get('https://trends.google.com/trends/?geo=VN',
                             timeout=REQUEST_TIMEOUT)
        except Exception as e:
            print(f"  [warn] Không lấy được cookie: {e}")

    # ---------- phương thức 1: pytrends ----------
    def _via_pytrends(self, keyword):
        if self.pytrends is None:
            raise RuntimeError('pytrends chưa khởi tạo')
        return self.pytrends.suggestions(keyword=keyword)

    # ---------- phương thức 2: request thủ công ----------
    def _via_manual(self, keyword):
        url = SUGGESTIONS_URL + quote(keyword, safe='')
        resp = self.session.get(url, params={'hl': HL, 'tz': TZ}, timeout=REQUEST_TIMEOUT)
        if resp.status_code == 429:
            raise requests.HTTPError('429 Too Many Requests', response=resp)
        resp.raise_for_status()
        # Google thêm tiền tố chống XSSI ")]}'," trước JSON
        text = re.sub(r"^\)\]\}',?\s*", '', resp.text)
        return json.loads(text)['default']['topics']

    # ---------- điều phối + fallback ----------
    def fetch(self, keyword):
        methods = [('pytrends', self._via_pytrends), ('manual', self._via_manual)]
        methods.sort(key=lambda m: m[0] != self.preferred)   # phương thức ưu tiên lên đầu
        last_error = None
        for method_name, method in methods:
            for attempt in range(1, MAX_RETRIES + 1):
                try:
                    topics = method(keyword)
                    self.preferred = method_name
                    return [t for t in (topics or []) if isinstance(t, dict)]
                except Exception as e:
                    last_error = e
                    wait = BACKOFF_BASE ** attempt + random.uniform(0, 1)
                    if _is_rate_limited(e):
                        wait *= RATE_LIMIT_MULTIPLIER
                    print(f"  [{method_name}] lần {attempt}/{MAX_RETRIES} lỗi: {e!r} -> chờ {wait:.1f}s")
                    time.sleep(wait)
                    if _is_rate_limited(e):
                        self._reset()          # đổi cookie/session khi bị chặn
            print(f"  [{method_name}] hết lượt thử, chuyển phương thức kế tiếp")
            self._reset()
        raise FetchError(f"{keyword!r}: {last_error!r}")

# 4. Chuẩn bị danh sách keyword

In [24]:
# export
keyword_sources = {
    'school': school_names,
    'major_group': major_group_names,
    'major': major_names,
    'admission_method': admission_method_names,
    'subject_group': subject_group_names,
    'subject': subject_names,
}

tasks = [(ktype, kw) for ktype, kws in keyword_sources.items() for kw in kws]

print(f"Tổng số keyword: {len(tasks)}")
print({k: len(v) for k, v in keyword_sources.items()})

Tổng số keyword: 1156
{'school': 302, 'major_group': 36, 'major': 451, 'admission_method': 13, 'subject_group': 274, 'subject': 80}


# 5. Lấy danh sách topic từ Google Trends

In [25]:
# export
rows = []
failed = []
no_result = []
fetcher = TopicFetcher()
consecutive_fails = 0

for i, (ktype, keyword) in enumerate(tasks, start=1):
    try:
        topics = fetcher.fetch(keyword)
        for topic in topics:
            rows.append({
                'keyword': keyword,
                'keyword_type': ktype,
                'mid': topic.get('mid'),
                'title': topic.get('title'),
                'type': topic.get('type'),
            })
        if not topics:
            no_result.append((ktype, keyword))
        consecutive_fails = 0
        print(f"[{i}/{len(tasks)}] ({ktype}) {keyword} -> {len(topics)} topic")
        time.sleep(random.uniform(*SLEEP_RANGE))
    except FetchError as e:
        failed.append((ktype, keyword))
        consecutive_fails += 1
        print(f"[{i}/{len(tasks)}] THẤT BẠI: {e}")
        if consecutive_fails >= CONSECUTIVE_FAIL_LIMIT:
            print(f"  Thất bại liên tiếp {consecutive_fails} lần -> nghỉ {COOLDOWN_SECONDS}s")
            time.sleep(COOLDOWN_SECONDS)
            fetcher._reset()
            consecutive_fails = 0

df_topics = pd.DataFrame(rows, columns=OUTPUT_COLUMNS)
df_topics = df_topics.drop_duplicates(subset=['keyword', 'keyword_type', 'mid'])

TOPICS_PATH.parent.mkdir(parents=True, exist_ok=True)
df_topics.to_csv(TOPICS_PATH, index=False, encoding='utf-8')
print(f"Hoàn tất. Số dòng: {len(df_topics)} | Không có topic: {len(no_result)} | Thất bại: {len(failed)}")
if failed:
    print("Keyword thất bại:", failed)

[1/1156] (school) Trường Cao Đẳng Kỹ Thuật Cao Thắng -> 3 topic
[2/1156] (school) Trường Cao Đẳng Công Thương TPHCM -> 4 topic
[3/1156] (school) Đại Học Kinh Tế Quốc Dân -> 3 topic
[4/1156] (school) Đại Học Bách Khoa Hà Nội -> 5 topic
[5/1156] (school) Học Viện Tài chính -> 4 topic
[6/1156] (school) Học Viện Ngân Hàng -> 3 topic
[7/1156] (school) Trường Đại học Ngoại thương -> 3 topic
[8/1156] (school) Trường Đại Học Bách Khoa HCM -> 4 topic
[9/1156] (school) Trường Đại Học Y Hà Nội -> 5 topic
[10/1156] (school) Học Viện Công Nghệ Bưu Chính Viễn Thông -> 5 topic
[11/1156] (school) Trường Đại Học Sư Phạm Hà Nội -> 5 topic
[12/1156] (school) Trường Đại Học Thương Mại -> 3 topic
[13/1156] (school) Trường Đại Học Sư Phạm TPHCM -> 4 topic
[14/1156] (school) Đại Học Y Dược TPHCM -> 2 topic
[15/1156] (school) Trường Đại Học Công Nghệ – Đại Học Quốc Gia Hà Nội -> 5 topic
[16/1156] (school) Trường Đại học Thủ Đô Hà Nội -> 3 topic
[17/1156] (school) Trường Đại Học Giao Thông Vận Tải -> 5 topic
[

# 6. Kiểm tra kết quả

In [26]:
# Khảo sát dữ liệu sau khi lưu
df_topics.info()
df_topics.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 3579 entries, 0 to 3578
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   keyword       3579 non-null   str  
 1   keyword_type  3579 non-null   str  
 2   mid           3579 non-null   str  
 3   title         3579 non-null   str  
 4   type          3579 non-null   str  
dtypes: str(5)
memory usage: 529.1 KB


,keyword,keyword_type,mid,title,type
0,Trường Cao Đẳng Kỹ Thuật Cao Thắng,school,/g/122jr9l6,Trường Cao đẳng Kỹ thuật Cao Thắng,"Trường cao đẳng nghề ở Thành phố Hồ Chí Minh, ..."
1,Trường Cao Đẳng Kỹ Thuật Cao Thắng,school,/g/11c6v9lswf,Thư viện trường Cao đẳng kỹ thuật Cao Thắng,"Thư viện đại học ở Thành phố Hồ Chí Minh, Việt..."
2,Trường Cao Đẳng Kỹ Thuật Cao Thắng,school,/g/11gw35hb5j,Ga Hàm Nghi,Chủ đề
3,Trường Cao Đẳng Công Thương TPHCM,school,/m/0vb4j37,Cao Đẳng Công Thương TP.HCM,"Cao đẳng ở Thành phố Hồ Chí Minh, Việt Nam"
4,Trường Cao Đẳng Công Thương TPHCM,school,/m/0vb3q34,Trường Đại học Công Thương TP. Hồ Chí Minh (HUIT),"Đại học ở Thành phố Hồ Chí Minh, Việt Nam"
5,Trường Cao Đẳng Công Thương TPHCM,school,/g/11bx2lgy79,Trường Cao đẳng Lý Tự Trọng TP. Hồ Chí Minh,Trường cao đẳng cộng đồng ở Thành phố Hồ Chí M...
6,Trường Cao Đẳng Công Thương TPHCM,school,/g/11c5szrcxf,Danh sách trường cao đẳng tại Thành phố Hồ Chí...,Chủ đề
7,Đại Học Kinh Tế Quốc Dân,school,/m/0bwj_9k,Trường Đại học Kinh tế Quốc dân,"Đại học công lập ở Hà Nội, Việt Nam"
8,Đại Học Kinh Tế Quốc Dân,school,/m/02q6bnb,Plekhanov Russian University of Economics,"Trường đại học ở Mát-xcơ-va, Nga"
9,Đại Học Kinh Tế Quốc Dân,school,/g/11h03bdz2j,Đại học Từ xa - Đại Học Kinh Tế Quốc Dân,"Trung tâm giáo dục ở Hà Nội, Việt Nam"


In [27]:
# Thống kê theo loại keyword và loại topic
print(df_topics.groupby('keyword_type')['keyword'].nunique())
print(df_topics['type'].value_counts().head(10))
print(f"Keyword không có topic: {len(no_result)}")

keyword_type
admission_method      5
major               422
major_group          32
school              268
subject              65
subject_group       135
Name: keyword, dtype: int64
type
Chủ đề                                              1264
Lĩnh vực nghiên cứu                               193
Trường đại học ở Hà Nội, Việt Nam                    115
Phần mềm                                             112
Phần mềm máy tính                                     83
Nghề nghiệp                                           79
Trường đại học ở Thành phố Hồ Chí Minh, Việt Nam      76
Dòng sản phẩm                                      69
Cơ quan chính phủ                                     55
Đại học ở Thành phố Hồ Chí Minh, Việt Nam           46
Name: count, dtype: int64
Keyword không có topic: 229


# 7. Xuất Notebook sang Unit Script

In [28]:
from helpers.export_to_script import export_to_script

export_to_script(output_path=SCRIPT_OUTPUT_PATH)

Đã xuất 7 cell từ 'get_topics.ipynb' sang 'D:\UNIDOCS\IS405.R11-Big-Data\scripts-fetching\unit-scripts\get_topics.py'


'D:\\UNIDOCS\\IS405.R11-Big-Data\\scripts-fetching\\unit-scripts\\get_topics.py'